## 🚀 Compound/Ensemble AI (vDAG) Load Testing

- Author: Shridhar Kini (Profile)
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run: `jupyter notebook --allow-root  --port 9999 --ip=0.0.0.0`
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace VDAGLoadTesting.ipynb`

This notebook provides an overview of the **Load Testing of VDAGs,** in the AIOS platform. These tutorial is designed to helps users to load test the VDAGs which is composed of AIOS building blocks, where each block represents a specific task or operation in the workflow. `Mux(fan in) & Demux(fan out), Sequential & Parallel AIOS blocks` concepts are also covered in this tutorial. 

### 📦 **Block Scaling and Load Testing**
- To understand the block scaling concepts in AIOS, please check following video tutorial: 
    - [14.PART-1: AIOS Block Scaling & Load Testing: Strategies, Policies, and Metrics With MidSized Model](https://www.youtube.com/watch?v=ZgU09r80TxA)
    - [14.PART-2: AIOS Block Scaling & Load Testing: Strategies, Policies, and Metrics with Smaller Model](https://www.youtube.com/watch?v=NS9vNDeoptc)

### 🏙️ **VDAGs in AIOS**
- VDAGs (Virtual Directed Acyclic Graphs) are a key abstraction in AIOS for representing complex workflows.
- VDAGs can be composed of multiple AIOS building blocks, each responsible for a specific task or operation.
- VDAGs enable efficient execution and scaling of workflows by allowing dynamic adjustment of resources based on workload.
- For more information on VDAGs, please refer to the below video tutorial:
    - [Part-1: Break Down Complex AI Models with AIOS v1's vDAG | A Deep Dive](https://www.youtube.com/watch?v=VROxR2e5RNE)
    - [Part-2: 09 vDAG Controller Policy Demonstration: Quota and Quality store policy](https://www.youtube.com/watch?v=OdBeVDoMhzE)
    - [Part-3: 09 vDAG Controllers Health Check Policy and Metrics](https://www.youtube.com/watch?v=XRc32ywSzX8)


### 🧪 **Load Testing Strategy**
- The load testing strategy includes the following steps:
    - Create a VDAG with the building blocks to be tested.
        - We used several high throughput open sourced LLM models to build the VDAG by creating multiple AIOS Blocks.
        - VDAG Has following Structure:
            - 1 Input block: Receives the input requests and generates the response to the query
            - 2 Review blocks: Each review block uses a smaller LLM model to review the response generated by the input block and provide feedback. And this will happen in parallel.
            - 1 Aggregator block: This block aggregates the responses from the review blocks and generates the final response.
                - ![VDAG](screenshots/vdag_diagram.png)

        - Each AIOS blocks are given with specialized system prompts in Chat interface to perform the specific tasks.
    - Understand the Models capabilities and limitations;
        - Evaluate the model size(Disk, RAM, CPU, GPU requirements), Tokens limits, Latency, Throughput, 
        - Example:
            - M1 : microsoft/Phi-4-mini-instruct
                - data type: fp16
                - Parameters: 3.84B
                - Disk: 7.2GB
                - GPU RAM: 8392MB
                - Context: 128K
                - Latency: 
                    - 1xA100:
                        - 1 instance: 1.98-2.89Sec (for 256 tokens with Batch of 4)
                        - 2 instance: 5.68-5.74Sec (for 256 tokens with Batch of 4)
                - Tokens/sec:
                    - 1xA100:
                        - 1 instance: 4*(89.5-90-94) tokens/sec (for Batch of 4) = 1*4*(89.5-90-94) = 358-376 tokens/sec
                        - 2 instance: 4*(44.97-46.04) tokens/sec (for Batch of 4) = 2*4*(44.97-46.04) = 359-368 tokens/sec
            - R1 : meta-llama/Llama-3.2-1B-Instruct
                - data type: fp16
                - Parameters: 1.23B
                - Disk: 4.7GB
                - GPU RAM: 3123MB
                - Context: 128K
                - Latency: 
                    - 1xA100:
                        - 1 instance: 1.25-1.275Sec (for 256 tokens with Batch of 4)
                        - 2 instance: 2.17-2.18Sec (for 256 tokens with Batch of 4)
                - Tokens/sec:
                    - 1xA100:
                        - 1 instance: 4*(203-205) tokens/sec (for Batch of 4) = 1*4*(203-205) = 812-820 tokens/sec
                        - 2 instance: 4*(118-119) tokens/sec (for Batch of 4) = 2*4*(118-119) = 944-952 tokens/sec
            - R2 : unsloth/Qwen3-1.7B
                - data type: fp16
                - Parameters: 1.7B
                - Disk: 3.3GB
                - GPU RAM: 4.2-4.3GB
                - Max Tokens: 32768
                - Latency: 
                    - 1xA100:
                        - 1 instance: 1.5-1.6Sec (for 256 tokens with Batch of 4)
                        - 2 instance: 2.9-3Sec (for 256 tokens with Batch of 4)
                    - 1xL4:
                        - 1 instance: 4.45-4.5Sec (for 256 tokens with Batch of 4)
                - Tokens/sec:
                    - 1xA100:
                        - 1 instance: 4*(162-164) tokens/sec (for Batch of 4) = 1*4*(162-164) = 648-656 tokens/sec
                        - 2 instance: 4*(87-188) tokens/sec (for Batch of 4) = 2*4*(87-88) = 696-704 tokens/sec
                    - 1xL4:
                        - 1 instance: 4*(57-58) tokens/sec (for Batch of 4) = 1*4*(57-58) = 228-232 tokens/sec
            - AGG : google/gemma-3-1b-it
                - data type: fp16
                - Parameters: 1B
                - Disk: 1.96GB
                - GPU RAM: 2805MB
                - Input Context: 32K
                - Output Context: 8K
                - Latency: 
                    - 1xA100:
                        - 1 instance: 1.5-1.51Sec (for 256 tokens with Batch of 4)
                        - 2 instance: 2.7-3.2Sec (for 256 tokens with Batch of 4)
                - Tokens/sec:
                    - 1xA100:
                        - 1 instance: 4*(174.4-175.7) tokens/sec (for Batch of 4) = 1*4*(174.4-175.7) = 698-702 tokens/sec
                        - 2 instance: 4*(96-97) tokens/sec (for Batch of 4) = 2*4*(96-97) = 768-776 tokens/sec

    - Total Latency Estimation for VDAG: Theoritical estimated based on the individual block latencies.
        - For 256 tokens/request:
            - Input Block (M1):     5.68-5.74Sec
            - Review Block 1 (R1):  2.17-2.18Sec
            - Review Block 2 (R2):
                - If A100: 2.9-3Sec
                - If L4: 4.45-4.5Sec
            - Aggregator Block (AGG): 2.7-3.2Sec
            - Total Estimated Latency for VDAG:
                - M1 + max(R1,R2) + AGG
                - = 5.68-5.74 + max(2.17-2.18,2.9-3/4.45-4.5) + 2.7-3.2
                - = 5.68-5.74 + 2.9-3/4.45-4.5 + 2.7-3.2
                - = 11.28-11.94Sec (If R2 on A100)
                - = 12.83-13.44Sec (If R2 on L4)
    
    - Understand Library used for running the inference:
        - Batching capability, Model Splitting capability (Like Tensor Parallelism, Pipeline Parallelism)etc.
            - Example: 
                - Library: vLLM
                - Batching: Yes (currently in our test 4 batch size is used)
                - Model Splitting: Yes (Tensor Parallelism)
    - Define the load testing scenarios:
        - Let us estimate the max request per second that can be handled by each blocks for given tokens generation per request.
        - For 256 tokens/request:
            - Input Block (M1):
                - 2 instance on 4xA100: 4*2*4*(44.97-46.04) = 1439.04-1473.28 tokens/sec
                    - i.e 1439.04-1473.28/256 = 5.621-5.755 requests/sec
            - Review Block 1 (R1):
                - 2 instance on 2xA100: 2*2*4*(118-119) = 1888-1904 tokens/sec
                    - i.e 1888-1904/256 = 7.375-7.4375 requests/sec
            - Review Block 2 (R2):
                - 2 instance on 1xA100: 1*2*4*(87-88) = 696-704 tokens/sec
                    - i.e 696-704/256 = 2.71-2.75 requests/sec
                - 1 instance on 4xL4: 4*1*4*(57-58) = 912-928 tokens/sec
                    - i.e 912-928/256 = 3.56-3.64 requests/sec
                - Total request/sec: 2 instance on 1xA100 + 1 instance on 4xL4:
                    - 696-704 + 912-928 = 1608-1632 tokens/sec
                    - i.e 1608-1632/256 = 6.28-6.37 requests/sec
            - Aggregator Block (AGG):
                - 2 instance on 2xA100: 2*2*4*(96-97) = 1536-1552 tokens/sec
                    - i.e 1536-1552/256 = 6-6.0625 requests/sec

            - Common request per sec from all 4 blocks:
                - Input Block (M1): 5.621-5.755 requests/sec
                - Review Block 1 (R1): 7.375-7.4375 requests/sec
                - Review Block 2 (R2): 6.28-6.37 requests/sec
                - Aggregator Block (AGG): 6-6.0625 requests/sec
                - So the common request per sec that can be handled by the VDAG is limited by the block with lowest request/sec capacity:
                    - i.e Input Block (M1): 5.621-5.755 requests/sec
            - For our load test we will be using 5 requests/sec
                - Every request is new session ID because we are testing with random questions in each requests.
                - Since we have fixed number of requests will be sent, we will not be testing the auto scaling capabilities of the blocks in this load test. i.e we will create all the instance required for the load test at the begining itself.
    - Concurancy or inflight request estimate:
        - For 256 tokens/request and estimated latency of 11.28-13.44Sec
        - If we want to send 5 requests/sec, then the concurancy or inflight requests will be:
            - 5 * 11.28-13.44 = 56.4-67.2 requests
        - So each block should be able to handle this concurancy or inflight requests.
    - Total instances and resources used in the load test:
        - Input Block (M1):
            - 2 instances on 4xA100 = 8 instances
        - Review Block 1 (R1):
            - 2 instances on 2xA100 = 4 instances
        - Review Block 2 (R2):
            - 2 instances on 1xA100 = 2 instances
            - 1 instance on 4xL4 = 4 instances
        - Aggregator Block (AGG):
            - 2 instances on 2xA100 = 4 instances
        - Total Instances:
            - A100: 18 instances
            - L4: 4 instances
        - Total GPU Used:
            - A100: 9 GPUs
            - L4: 4 GPUs
    - Logging/Monitoring:
        - Log the request and response times metrics to DB.
            - We will be using TimescaleDB to storing the metrics.
            - Custom scripts to visualize above metrics
        - Monitor the resource utilization and metrics of the block instances.
            - We will be using Prometheus and Grafana for monitoring and visualization.
    - Monitor the block performance during the load test
        - check for latency, error rates, queue lengths, total request processed etc.


**To  GET PORT MAPPING wrt to Service**([Doc](https://docs.aigr.id/installation/installation/#deploying-registry-services))

In [ ]:
# 🔧 Configuration Setup - Run this cell first to set up shared variables
import os

# Set configuration variables that will be available across all cells
GATEWAY_URL = "MANAGEMENTMASTER:30600"
CLUSTER_ID = "gcp-cluster-2"
GLOBAL_CLUSTER_METRICS_DB = "MANAGEMENTMASTER:30202"
GLOBAL_BLOCK_METRICS_DB = "MANAGEMENTMASTER:30201"
PARSER_URL = "MANAGEMENTMASTER:30501"
GLOBAL_CLUSTER_DB = "MANAGEMENTMASTER:30101"
GLOBAL_TASK_DB_SERVICE = "MANAGEMENTMASTER:30108"
COMPONENT_REGISTRY_SERVICE = "MANAGEMENTMASTER:30112"
GLOBAL_BLOCKDB_SERVICE = "MANAGEMENTMASTER:30100"
VDAG_DB_SVC = "MANAGEMENTMASTER:30103"
GLOBAL_VDAG_METRICS = "MANAGEMENTMASTER:30203"  # For other API calls

# Set environment variables for bash cells
os.environ['GATEWAY_URL'] = GATEWAY_URL
os.environ['CLUSTER_ID'] = CLUSTER_ID
os.environ['GLOBAL_CLUSTER_METRICS_DB'] = GLOBAL_CLUSTER_METRICS_DB
os.environ['GLOBAL_BLOCK_METRICS_DB'] = GLOBAL_BLOCK_METRICS_DB
os.environ['PARSER_URL'] = PARSER_URL
os.environ['GLOBAL_CLUSTER_DB'] = GLOBAL_CLUSTER_DB
os.environ['GLOBAL_TASK_DB_SERVICE'] = GLOBAL_TASK_DB_SERVICE
os.environ['COMPONENT_REGISTRY_SERVICE'] = COMPONENT_REGISTRY_SERVICE
os.environ['GLOBAL_BLOCKDB_SERVICE'] = GLOBAL_BLOCKDB_SERVICE
os.environ['VDAG_DB_SVC'] = VDAG_DB_SVC
os.environ['GLOBAL_VDAG_METRICS'] = GLOBAL_VDAG_METRICS

print("✅ Configuration variables set:")
print(f"   • GATEWAY_URL: {GATEWAY_URL}")
print(f"   • CLUSTER_ID: {CLUSTER_ID}")
print(f"   • GLOBAL_CLUSTER_METRICS_DB: {GLOBAL_CLUSTER_METRICS_DB}")
print("\n📝 These variables are now available in both Python and bash cells!")
print("   - In Python: use GATEWAY_URL, CLUSTER_ID, GLOBAL_CLUSTER_METRICS_DB PARSER_URL GLOBAL_CLUSTER_DB GLOBAL_TASK_DB_SERVICE")
print("   - In bash: use $GATEWAY_URL, $CLUSTER_ID, $GLOBAL_CLUSTER_METRICS_DB $PARSER_URL $GLOBAL_CLUSTER_DB $GLOBAL_TASK_DB_SERVICE")
os.system('echo $GATEWAY_URL')
os.system('echo $CLUSTER_ID')
os.system('echo $GLOBAL_CLUSTER_METRICS_DB')
os.system('echo $PARSER_URL')
os.system('echo $GLOBAL_CLUSTER_DB')
os.system('echo $GLOBAL_TASK_DB_SERVICE')
os.system('echo $COMPONENT_REGISTRY_SERVICE')
os.system('echo $GLOBAL_BLOCKDB_SERVICE')
os.system('echo $GLOBAL_BLOCK_METRICS_DB')
os.system('echo $VDAG_DB_SVC')

### **Resource Allocation Policy** [Code](policies/resource_allocator)

##### Register the component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
 -d @./policies/resource_allocator/registration.json \
 -H "Content-Type: application/json" | json_pp

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.scaletest_block_resource_allocator:1.0.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/resource_allocator/upload.sh

### **Load Balancer Policy** [Code](policies/weighted_metrics_load_balancer)

##### Register the component

In [ ]:
%%bash
bash policies/weighted_metrics_load_balancer/register.sh

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.weightedmetricsloadbalancer:2.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/weighted_metrics_load_balancer/upload.sh

### **Auto-scaling Policy** [Code](policies/queue_length_scaling_metrics)

##### Register the component

In [ ]:
%%bash
bash policies/queue_length_scaling_metrics/register.sh

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.queuebasedautoscaler:2.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/queue_length_scaling_metrics/upload.sh

### **Health Check Policy** [Code](policies/block_healthcheck_policy)

##### Register the component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
 -d @./policies/block_healthcheck_policy/block_health_check_registration.json \
 -H "Content-Type: application/json" | json_pp

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.demo_block_health_checker:1.0.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/block_healthcheck_policy/upload.sh

### **Create Blocks in Cluster(With Above Policies)** [Code](model_code)

**Important Notes:**
- Currently Batching and Muxing is handled in Block Code itself.
- In General Batching should be handled in Block Code and Muxing should be handled in preprocessing(i.e `preprocessingPolicyRule`) of Block in VDAG. 

#### **Build Docker Image**

In [ ]:
%%bash
bash model_code/build_docker_vllm.sh

#### **Push Docker Image to Registry**

In [ ]:
%%bash
docker push MANAGEMENTMASTER:31280/vllm_batching_aios:v1

#### **Register Components**

In [ ]:
%%bash 
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
  -H "Content-Type: application/json" \
  -d @./blocks/mistrall_vllm/component.json | json_pp

#### **Unregister Components**

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.magistral-small-2506-vllm:1.0.0-stable"}' | json_pp

#### **Deploy All the Blocks**

##### **Create First Instance of Each Block**

In [ ]:
%%bash
bash blocks/manually_allocate_first_instance.bash

##### **Scale Instances of Each Block as per Load Test Plan**

In [ ]:
%%bash
bash blocks/manual_scaling_for_vdag.bash

#### **Get Block Metrics**

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/phi-4-mini-instruct-vllm-block \
    -H "Content-Type: application/json" | json_pp

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/llama-3-2-1b-instruct-vllm-block \
    -H "Content-Type: application/json" | json_pp

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/qwen3-1-7b-vllm-block \
    -H "Content-Type: application/json" | json_pp

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/gemma-3-1b-it-vllm-block \
    -H "Content-Type: application/json" | json_pp

#### **Create Inference Server For The Cluster**
- Run this commands in your cluster node(like master node)

- `kubectl create namespace inference-server`
- `kubectl create -f inference_server/inference_server.yaml`

#### **Create the VDAG By connecting the blocks**

In [ ]:
import requests

PARSER_URL_VDAGCREATION = "http://MANAGEMENTMASTER:30501/api/createvDAG"

data = {
  "parser_version": "Parser/V1",
  "body": {
    "spec": {
      "values": {
        "vdagName": "scale-test-vdag-2",
        "vdagVersion": {
          "version": "0.0.1",
          "release-tag": "stable"
        },
        "discoveryTags": [
          "scale-vdag-llm",
          "scale-llm-vdag"
        ],
        "controller": {},
        "nodes": [
          {   
            "spec": {
              "values": {
                "nodeLabel": "agg_gemma",
                "nodeType": "block",
                "manualBlockId": "gemma-3-1b-it-vllm-block",
                "preprocessingPolicyRule": {},
                "postprocessingPolicyRule": {
                  
                },
                "modelParameters": {}
              },
              "IOMap": [
                {
                  "inputs": [
                    {
                      "name": "input_0",
                      "reference": "input_0"
                    }
                  ],
                  "outputs": [
                    {
                      "name": "output_0",
                      "reference": "output_0"
                    }
                  ]
                }
              ]
            }
          },
          {
            "spec": {
              "values": {
                "nodeLabel": "r1_llama3",
                "nodeType": "block",
                "manualBlockId": "llama-3-2-1b-instruct-vllm-block",
                "preprocessingPolicyRule": {},
                "postprocessingPolicyRule": {},
                "modelParameters": {}
              },
              "IOMap": [
                {
                  "inputs": [
                    {
                      "name": "input_0",
                      "reference": "input_0"
                    }
                  ],
                  "outputs": [
                    {
                      "name": "output_0",
                      "reference": "output_0"
                    }
                  ]
                }
              ]
            }
          },
          {
            "spec": {
              "values": {
                "nodeLabel": "r2_qwen3",
                "nodeType": "block",
                "manualBlockId": "qwen3-1-7b-vllm-block",
                "preprocessingPolicyRule": {},
                "postprocessingPolicyRule": {
                  
                },
                "modelParameters": {}
              },
              "IOMap": [
                {
                  "inputs": [
                    {
                      "name": "input_0",
                      "reference": "input_0"
                    }
                  ],
                  "outputs": [
                    {
                      "name": "output_0",
                      "reference": "output_0"
                    }
                  ]
                }
              ]
            }
          },
          
          {
            "spec": {
              "values": {
                "nodeLabel": "m1_phi4_mini",
                "nodeType": "block",
                "manualBlockId": "phi-4-mini-instruct-vllm-block",
                "preprocessingPolicyRule": {},
                "postprocessingPolicyRule": {
                  
                },
                "modelParameters": {}
              },
              "IOMap": [
                {
                  "inputs": [
                    {
                      "name": "input_0",
                      "reference": "input_0"
                    }
                  ],
                  "outputs": [
                    {
                      "name": "output_0",
                      "reference": "output_0"
                    }
                  ]
                }
              ]
            }
          }
        ],
        "graph": {
          "input": [
            {
              "nodeLabel": "m1_phi4_mini",
              "inputNames": [
                "input_0"
              ]
            }
          ],
          "output": [
            {
              "nodeLabel": "agg_gemma",
              "outputNames": [
                "output_0"
              ]
            }
          ],
          "connections": [
            {
              "nodeLabel": "r1_llama3",
              "inputs": [
                {
                  "nodeLabel": "m1_phi4_mini",
                  "outputNames": [
                    "output_0"
                  ]
                }
              ]
            },
            {
              "nodeLabel": "r2_qwen3",
              "inputs": [
                {
                  "nodeLabel": "m1_phi4_mini",
                  "outputNames": [
                    "output_0"
                  ]
                }
              ]
            },
            {
              "nodeLabel": "agg_gemma",
              "inputs": [
                {
                  "nodeLabel": "r1_llama3",
                  "outputNames": [
                    "output_0"
                  ]
                },
                {
                  "nodeLabel": "r2_qwen3",
                  "outputNames": [
                    "output_0"
                  ]
                }
              ]
            }
          ]
        }
      }
    }
  }
}

response = requests.post(PARSER_URL_VDAGCREATION, json=data)
print(response.status_code)
print('api response', response.json())

**You can confirm the creation of vDAG by querying the vDAG information using `vdagURI` from vDAGs registry**:

In [2]:
%%bash
curl -X GET http://$VDAG_DB_SVC/vdag/scale-test-vdag-2:0.0.1-stable | json_pp

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3478  100  3478    0     0  37697      0 --:--:-- --:--:-- --:--:-- 37397


{
   "data" : {
      "assignment_info" : {
         "agg_gemma" : "gemma-3-1b-it-vllm-block",
         "m1_phi4_mini" : "phi-4-mini-instruct-vllm-block",
         "r1_llama3" : "llama-3-2-1b-instruct-vllm-block",
         "r2_qwen3" : "qwen3-1-7b-vllm-block"
      },
      "compiled_graph_data" : {
         "head" : "phi-4-mini-instruct-vllm-block",
         "rev_mapping" : {
            "gemma-3-1b-it-vllm-block" : "agg_gemma",
            "llama-3-2-1b-instruct-vllm-block" : "r1_llama3",
            "phi-4-mini-instruct-vllm-block" : "m1_phi4_mini",
            "qwen3-1-7b-vllm-block" : "r2_qwen3"
         },
         "t2_graph" : {
            "gemma-3-1b-it-vllm-block" : [],
            "llama-3-2-1b-instruct-vllm-block" : [
               "gemma-3-1b-it-vllm-block"
            ],
            "phi-4-mini-instruct-vllm-block" : [
               "llama-3-2-1b-instruct-vllm-block",
               "qwen3-1-7b-vllm-block"
            ],
            "qwen3-1-7b-vllm-block" : [
         

## 🧭 Step 2: Deploy a vDAG Controller

The vDAG Controller is the **runtime engine** that orchestrates the flow of data through the vDAG graph.

It handles:
- Task routing between blocks
- Health and quota monitoring
- Quality management - using a policy to capture outputs and verifiying (manual/automated)

You can read about the vDAG controller [here](https://docs.aigr.id/vdag-controller/vdag-controller/).

### Configuration Parameters:
- `vdag_uri`: Which vDAG this controller will serve
- `policy_execution_mode`: Whether to run policies locally or remotely
- `replicas`: How many controller pods to run (for redundancy or scale)

> You can deploy multiple controllers for the same vDAG across clusters for multi-region or HA setups.

The command below deploys a vDAG controller for the vDAG we created with 1 replicas:

In [ ]:
%%bash
curl -X POST http://$GATEWAY_URL/vdag-controller/gcp-cluster-2 \
  -H "Content-Type: application/json" \
  -d '{
    "action": "create_controller",
    "payload": {
      "vdag_controller_id": "scaletest-vdag-controller", 
      "vdag_uri": "scale-test-vdag-2:0.0.1-stable",
      "config": {
        "policy_execution_mode": "local",
        "replicas": 1
      },
      "search_tags": []
    }
  }'


**We can query the available controllers for the given vDAG using the command below by specifying the `vDAGURI`**

In [ ]:
%%bash
curl -X GET http://$VDAG_DB_SVC/vdag-controllers/by-vdag-uri/scale-test-vdag-2:0.0.1-stable | json_pp

**The individual controller details can also be queried by specifying the `vdag_controller_id`**

In [ ]:
%%bash
curl -X GET http://$VDAG_DB_SVC/vdag-controller/scaletest-vdag-controller | json_pp

Every vDAG controller exposes REST and gRPC APIs for submitting inference requests and a REST service for management of health checker, quality checker and quota management policies. `config.api_url` can be used to submit inference requests using REST API and `config.rpc_url` can be used for submitting inference requests using GRPC interface.

## 🤖 Step 3: Run Inference on the vDAG

We now simulate an **inference task** using a multi-modal input — both **text** and **image**.

### Input Structure:
- `session_id`: Used for tracking and quota enforcement
- `seq_no`: Monotonically increasing number per session
- `data.mode`: Set to `"chat"` to enable conversational behavior
- `messages`: The core input — includes both a text prompt and an image URL

### What Happens:
1. The request is received by the controller
2. It routes the input to `M1`
3. Then the response from `M1` to `R1` and `R2`
4. Finally the response of `R1` and `R2` are sent to `AGG` at different intervals of time where both responses are passed to AGG blocks instance to generate the final output.


### Inference using REST API

Inference requests can be submitted using the REST API using the command below (the REST API url can be obtained from `config.api_url` field of vDAG controller data):


In [ ]:
%%bash
curl -X POST  http://CLUSTER2MASTER:31126/v1/infer \
  -H "Content-Type: application/json" \
  -d '{
  "retries": 3,
  "timeout": 1200,
  "session_id": "session1",
  "seq_no": 2,
  "data": {
    "mode": "generate",
    "generation_config": {
      "temperature": 0.7,
      "repetition_penalty": 1.0,
      "min_p": 0.01,
      "top_k": -1,
      "top_p": 0.95,
      "max_tokens": 256
    },
    "prompt": "write a one lines about photosynthesis"
  },
  "graph": {},
  "selection_query": {}
}' | json_pp


### **VDAG METRICS**

In [6]:
%%bash
curl -X GET http://$GLOBAL_VDAG_METRICS/vdag/scaletest-vdag-controller | json_pp

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   517  100   517    0     0   4735      0 --:--:-- --:--:-- --:--:--  4743


{
   "data" : {
      "inference_fps" : 0.106284433983791,
      "inference_latency_seconds" : 9.40871548652649,
      "inference_requests_total" : 436124,
      "inference_retries_total" : 0,
      "inflight_requests" : 53,
      "timestamp" : 1761303534.47798,
      "total_complete_total" : 436124,
      "total_retries_0_total" : 434748,
      "total_retries_1_total" : 1376,
      "total_retries_2_total" : 0,
      "total_retries_plus_total" : 0,
      "total_retries_total" : 1376,
      "type" : "vdag",
      "vdagControllerId" : "scaletest-vdag-controller",
      "vdagURI" : "scale-test-vdag-2:0.0.1-stable"
   },
   "success" : true
}


## 🧹 Step 4: Clean-up

### **The controller can be removed using the following command**

In [ ]:
%%bash
curl -X POST http://$GATEWAY_URL/vdag-controller/gcp-cluster-2 \
  -H "Content-Type: application/json" \
  -d '{
    "action": "remove_controller",
    "payload": {
      "vdag_controller_id": "scaletest-vdag-controller"
    }
  }'

### **The vDAG entry if not needed anymore can be removed using the following command:**

In [ ]:
%%bash
curl -X DELETE http://$VDAG_DB_SVC/vdag/scale-test-vdag-2:0.0.1-stable


### **Remove all the deployed blocks using the following commands:**

In [ ]:
%%bash
bash blocks/remove_all_blocks.bash

### 📊 **Run Response Time and Failures Visualization** [Code](streamlit_visualize_response_time.py)

- `bash setup_timescaledb_env.sh`
    - To Setup all env variables required to connect to TimescaleDB

In [ ]:
# Run this in screen, So that this UI is always available 
bash run_visualize_response_time.sh

- to get streamlit UI:
    - In browser : http://SERVERIP:8502/   
        - Port and IP will vary based on your setup
- Use the test_id generated from the `vdag_load_test.py` script to visualize the response time and failures. Filter the data for selected timeranges as shown below.

    ##### Average Response Time Graph (averaged over 60 seconds)
    ![Average Response Time Graph (averaged over 60 seconds)](screenshots/vdag_streamlit1.png)

    ##### Input Request Rate (averaged over 60 seconds)
    ![Input Request Rate (averaged over 60 seconds)](screenshots/vdag_streamlit2.png)

    ##### Total Request Count
    ![Total Request Count](screenshots/vdag_streamlit3.png)

    ##### Table View of Requests
    ![Table View of Requests](screenshots/vdag_streamlit4.png)

### 📉 **Create Grafana Dashboard for Blocks metrics** [Dashboard Json](blocks/Sample_block_grafana_dashboard.json)

- Add new Data Source in Grafana
    - Name: `Prometheus`
    - Prometheus Server URL: `http://prometheus-server.metrics-storage.svc.cluster.local`
    - Open http://MANAGEMENTMASTER:32199/connections/datasources
- Open the Grafana Dashboard using the URL: http://MANAGEMENTMASTER:32199/dashboards
- Create a new dashboard and import the above json file.
    - Press `New` -> `Import` -> Upload the above json file -> Press `Import` button
    - Use the json provided in `blocks/Sample_block_grafana_dashboard.json`
- Now open the dashboard to see the metrics.
    - If any time block ID changes, update the `blocks/Sample_block_grafana_dashboard.json` with new block ID or edit the panels to use the new block ID.
    - i.e replace `llama-3-2-1b-instruct-vllm-block` to your block ID.
    - replace `"title": "VLLM Llama-3.2-1B-Instruct"`  to `"title": "VLLM YOURBLOCKID"`,
    - replace `"uid": "qwenaezthfgdsvsd"` to `"uid": "dummy random string"`,
- Create a dashboard for VDAG Controllers Metrics as well
    - use `blocks/vdag_sample_grafana_dashboard.json`
- Attached Few Screenshots of the VDAG dashboard below:
    - ![Dashboard1](screenshots/vdag_grafana_dashbaord1.png)

- Attached Few Screenshots of the Blocks dashboard below:
    - ![Dashboard1](screenshots/phi_grafana_1.png)
    - ![Dashboard2](screenshots/phi_grafana_2.png)
    - ![Dashboard3](screenshots/phi_grafana_3.png)
    - ![Dashboard4](screenshots/phi_grafana_4.png)
    - ![Dashboard5](screenshots/phi_grafana_5.png)

### 👥 **Run VDAG Load Testing Script**

- `vdag_load_test.py` file contains the code to simulate the vdag inference with given request per second.
- To fetch random question, use script `generate_questions.py` and uses GEMINI API to fetch random questions. You may have to supply the API key in the script or ENV variable.
    - We have already generated 2.1k+ questions and stored in `questions.jsonl` file.
    - each session_ids will pick the questions from here, once all questions are over, again it starts taking questions from this file randomly. Randomness is added so that same question is not picked up by multiple sessions at the same time, such they they end up in same instance.
- install dependencies:
    - `pip install -r requirements.txt`
    - `pip install -r requirements_streamlit.txt`
    - `bash setup_timescaledb_env.sh`
        - To Setup all env variables required to connect to TimescaleDB
- get the vDAG controller API URL from the vDAG controller created in Step 2. i.e `config.api_url` and update in `config.yaml` file.
- To run the code, use the command: `python3 vdag_load_test.py`
- better to run this code in screen  as this will run for longer duration configured in config.yaml file.
- Upon each request it calls the vdag endpoint and logs the response time and failure time to DMA Endpoint DB.
    - ```json{
        "block_id":  "dummy",
        "session_id": session.session_id,
        "seq_no": seq_no,
        "type": "success",
        "response_time": 0.0,
        "raw": "{}",
        "test_id": self.test_id,
        "user_id": "dummy",
        "starttime": time.time(),
        "endtime": time.time(),
        "starttimeObj": datetime.now(self.ist_tz),
        "endtimeObj": datetime.now(self.ist_tz)
    } ```
- It Divides Users request with equidistant time interval based on the request rate configured in config.yaml file. 
    - i.e 5 requests per second means one request every 200 milliseconds.

## **Conclusion**
- AI SCALE is performing at theoretical efficiency.
    - Latency perspective:
        - Theoritical Estimated Latency for VDAG:
            - = 11.28-11.94Sec (If R2 on A100)
            - = 12.83-13.44Sec (If R2 on L4)
        - Observed Latency for VDAG from Load Test:
            - = 8-11Sec
    - Request per second perspective:
        - 5.621-5.755 requests/sec is the estimated common request per sec that can be handled by the VDAG.
        - Load test is performed at 5 requests/sec and the VDAG is able to handle the load.
    - Inflight requests perspective:
        - Estimated Inflight requests: 56.4-67.2 requests
        - Observed Inflight requests during load test: 48-51 requests
        - This confirms synchronous DAG orchestration and scaling logic are correctly tuned.
    - Failure percentage:
        - Failed request in first attempts: 1148
        - Total requests sent: 311k
        - Failure percentage: 0.36%

- Next Steps:
    - Handle a bug in load balancer which fails in Mux block at certain interval of time.
    - Integrate automated quality checks in the VDAG workflow.
    - handle muxing in preprocessing of block in VDAG rather than in block code itself.